## Preprocessing

## 1.Initial setup

In [17]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw/utkface/UTKFace_aligned_cropped/UTKFace")

Read the metadata df

In [ ]:
df = pd.read_csv("../data/metadata/utkface_metadata.csv")

# make sure paths are Path objects
df["image_path"] = df["image_path"].apply(str)

#verify
print(df.shape)
df.info()





(23705, 5)
<class 'pandas.DataFrame'>
RangeIndex: 23705 entries, 0 to 23704
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   image_path  23705 non-null  str  
 1   age         23705 non-null  int64
 2   gender      23705 non-null  int64
 3   race        23705 non-null  int64
 4   age_group   23705 non-null  str  
dtypes: int64(3), str(2)
memory usage: 926.1 KB


Filter out ages <10 and >100

In [29]:
df = df[df["age"].between(10, 100)]


df = df.reset_index(drop=True)
print("Remaining samples:", len(df))

Remaining samples: 20622


In [30]:
df["age"].describe()

count    20622.000000
mean        37.664388
std         17.195798
min         10.000000
25%         26.000000
50%         32.000000
75%         48.000000
max        100.000000
Name: age, dtype: float64

## 2. Creating Split

In [31]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["age_group"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["age_group"],
    random_state=42
)

In [32]:
train_df["age_group"].value_counts(normalize=True)
val_df["age_group"].value_counts(normalize=True)
test_df["age_group"].value_counts(normalize=True)

age_group
young     0.430511
middle    0.393988
old       0.175501
Name: proportion, dtype: float64

We only have 17% of of "old"- this should be enough for the learning. If not we will use data augmentation or other methods when training. 

Check if proportions are similar

In [33]:
print(train_df["age_group"].value_counts(normalize=True))
print(val_df["age_group"].value_counts(normalize=True))
print(test_df["age_group"].value_counts(normalize=True))


age_group
young     0.430343
middle    0.394042
old       0.175615
Name: proportion, dtype: float64
age_group
young     0.430327
middle    0.394116
old       0.175558
Name: proportion, dtype: float64
age_group
young     0.430511
middle    0.393988
old       0.175501
Name: proportion, dtype: float64


Save this split

In [35]:
train_df.to_csv("../data/metadata/train.csv", index=False)
val_df.to_csv("../data/metadata/val.csv", index=False)
test_df.to_csv("../data/metadata/test.csv", index=False)